# OTT Streaming Content Trend Analysis
**Veda Technology – Task 32 | Python EDA + SQL + Business Memo**

This notebook explores an OTT catalog dataset and produces charts for content mix, genre popularity, release trends, ratings, countries and duration.  
**Dataset note:** The included `ott_titles_sample.csv` is a reproducible synthetic practice dataset (650 records), not an official Netflix/Prime/Hotstar export. Replace it with the public Kaggle Netflix Movies and TV Shows dataset to perform analysis on the original catalog. The schema is compatible with the common Netflix titles dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "ott_titles_sample.csv"
df = pd.read_csv(RAW)
df.head()

## 1. Data quality and overview

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print("Duplicate show IDs:", df.duplicated(subset=["show_id"]).sum())

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
for col in ["country", "rating", "director", "cast"]:
    df[col] = df[col].fillna("Unknown")
df["added_year"] = df["date_added"].dt.year
df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\\d+)")[0], errors="coerce")
df["duration_unit"] = df["duration"].str.extract(r"(min|Season)", expand=False).fillna("Unknown")
df = df.drop_duplicates(subset=["show_id"])
print("Cleaned shape:", df.shape)

## 2. Exploratory visualizations
All charts are generated from the included data.

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7,4))
sns.countplot(data=df, x="type", order=df["type"].value_counts().index, ax=ax)
ax.set(title="1. Movies vs TV Shows", xlabel="Content type", ylabel="Number of titles")
plt.tight_layout(); plt.show()

In [ ]:
genre_counts = df["listed_in"].str.split(", ").explode().value_counts().head(10)
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(x=genre_counts.values, y=genre_counts.index, ax=ax)
ax.set(title="2. Top 10 genres/categories", xlabel="Number of titles", ylabel="Genre")
plt.tight_layout(); plt.show()

In [ ]:
year_counts = df.groupby("release_year").size().sort_index()
fig, ax = plt.subplots(figsize=(10,4))
year_counts.plot(ax=ax)
ax.set(title="3. Titles by release year", xlabel="Release year", ylabel="Number of titles")
plt.tight_layout(); plt.show()

In [ ]:
rating_counts = df["rating"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9,4))
sns.barplot(x=rating_counts.index, y=rating_counts.values, ax=ax)
ax.set(title="4. Content rating distribution (top 10)", xlabel="Rating", ylabel="Number of titles")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

In [ ]:
country_counts = df["country"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(x=country_counts.values, y=country_counts.index, ax=ax)
ax.set(title="5. Top 10 countries represented", xlabel="Number of titles", ylabel="Country")
plt.tight_layout(); plt.show()

In [ ]:
movies = df[(df["type"]=="Movie") & (df["duration_unit"]=="min")].copy()
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(movies["duration_value"].dropna(), bins=18, kde=True, ax=ax)
ax.set(title="6. Movie runtime distribution", xlabel="Runtime (minutes)", ylabel="Number of movies")
plt.tight_layout(); plt.show()

In [ ]:
added = df.dropna(subset=["added_year"]).groupby("added_year").size()
fig, ax = plt.subplots(figsize=(8,4))
added.plot(kind="bar", ax=ax)
ax.set(title="7. Catalog additions by year", xlabel="Year added", ylabel="Titles added")
plt.tight_layout(); plt.show()

## 3. Business questions and findings
1. Which content format dominates the catalog?
2. Which categories appear most often?
3. How does the release-year profile vary?
4. Which audience ratings are most common?
5. Which countries contribute the largest number of titles?
6. What is the typical movie runtime distribution?

**Interpretation caution:** This bundled dataset is synthetic and is intended to validate the workflow and charting code. Do not present its numeric findings as facts about any streaming platform.

In [ ]:
summary = {
    "total_titles": len(df),
    "movies": int((df["type"]=="Movie").sum()),
    "tv_shows": int((df["type"]=="TV Show").sum()),
    "top_genre": df["listed_in"].value_counts().index[0],
    "top_country": df["country"].value_counts().index[0],
    "median_movie_runtime_min": float(movies["duration_value"].median())
}
summary

## 4. Export analysis-ready data
Run this cell to save a cleaned CSV for Excel pivot tables or SQL loading.

In [ ]:
OUT = ROOT / "data" / "processed" / "ott_titles_cleaned.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print("Exported:", OUT)